# Phase 3: Runtime Species Configuration

Verifies that the mechanism can be switched at runtime (config-only, no recompilation).
Compares species sets between Chapman and analytical mechanisms, and confirms Chapman
produces bitwise-identical results to Phase 2.

**Pre-requisite:**
- `data/jw_480km_chapman/output.nc` (run with Chapman config)
- `data/jw_480km_analytical/output.nc` (run with analytical config)

In [ ]:
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("..") / "data"
CHAPMAN = DATA_DIR / "jw_480km_chapman" / "output.nc"
ANALYTICAL = DATA_DIR / "jw_480km_analytical" / "output.nc"
PHASE2_REF = DATA_DIR / "jw_480km_chapman" / "output.nc"  # same file for Phase 2 baseline

## 1. Species Comparison

Each mechanism defines different species. The runtime allocator should
create exactly the right tracer set for each config.

In [ ]:
def get_chem_species(path):
    """Return chemistry species variable names from a NetCDF file."""
    if not path.exists():
        return None
    ds = nc.Dataset(path)
    # Heuristic: chemistry species have (Time, nCells, nVertLevels) shape
    exclude = {"pressure_base", "pressure_p", "theta", "uReconstructZonal",
               "uReconstructMeridional", "latCell", "lonCell", "surface_pressure"}
    species = [v for v in ds.variables if v not in exclude
               and len(ds[v].dimensions) == 3]
    ds.close()
    return species

for label, path in [("Chapman", CHAPMAN), ("Analytical", ANALYTICAL)]:
    sp = get_chem_species(path)
    if sp is None:
        print(f"{label}: file not found — Phase 3 not yet run")
    else:
        print(f"{label}: {len(sp)} species — {sp}")

## 2. Bitwise Comparison with Phase 2

The Phase 3 Chapman run should produce identical output to Phase 2
(same binary, same config = same results).

In [ ]:
if CHAPMAN.exists() and PHASE2_REF.exists():
    ds1 = nc.Dataset(CHAPMAN)
    ds2 = nc.Dataset(PHASE2_REF)
    all_match = True
    for v in ds1.variables:
        if v in ds2.variables:
            diff = np.abs(ds1[v][:] - ds2[v][:]).max()
            if diff > 0:
                print(f"  DIFF {v}: max abs diff = {diff:.2e}")
                all_match = False
    ds1.close()
    ds2.close()
    print(f"\n  Bitwise match: {'YES' if all_match else 'NO'}")
else:
    print("Files not available for comparison")

In [ ]:
print("Phase 3: requires both Chapman and analytical mechanism runs")
print("See verification/README.md for instructions")